# Interest rate risk in the banking book of a Swiss cantonal bank (BCV)

A short walkthrough. The full analysis is in `scripts/`, the report in `report/`.
Every number comes from public documents saved in `data/` (BCV Pillar 3 and annual report 2025,
FINMA Circular 2019/2, SNB data portal).

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from src import data, shocks
from src.bcv import build_positions
from src.curve import ZeroCurve
from src.model import delta_eve, delta_nii

## 1. What BCV publishes
The Pillar 3 report gives, for the parent bank at 31 Dec 2025, the change in economic value of
equity (ΔEVE) and in net interest income (ΔNII) under the six FINMA shocks (table IRRBB1).

In [2]:
published = data.bcv_published_irrbb1()
latest = published[published.date == "2025-12-31"].set_index("scenario")
tier1 = data.key_figures()["tier1_parent"]["value"]
latest[["delta_eve_chf_m", "delta_nii_chf_m"]]

,delta_eve_chf_m,delta_nii_chf_m
scenario,,
parallel_up,-467,78.0
parallel_down,488,-3.0
steepener,-138,NaN
flattener,40,NaN
short_up,-120,NaN
short_down,125,NaN


## 2. The balance sheet behind it
Table IRRBBA1 gives the volume and the average repricing date of each block of the banking book.

In [3]:
positions_table = data.bcv_positions()
positions_table[["item", "volume_chf_m", "avg_repricing_y", "max_repricing_y"]]

,item,volume_chf_m,avg_repricing_y,max_repricing_y
0,interbank_claims,525,0.8,NaN
1,customer_loans_fixed,5212,3.3,NaN
2,mortgages_money_market,455,0.0,NaN
3,mortgages_fixed,34178,4.0,NaN
4,financial_investments,7337,6.0,NaN
5,swap_receive_legs,12887,0.3,NaN
6,interbank_liabilities,-4147,0.2,NaN
7,customer_term_deposits,-6656,0.3,NaN
8,cash_bonds,-4,1.2,NaN
9,pfandbrief_loans,-11238,6.8,NaN


## 3. The CHF curve and the six shocks
Svensson curve fitted to 22 Confederation bonds and SARON on 30 Dec 2025 (`scripts/02_curve_and_inputs.py`).
FINMA sizes for CHF: parallel 150 bp, short 150 bp, long 100 bp, no floor.

In [4]:
curve_info = json.loads((ROOT / "results" / "curve.json").read_text())
curve = ZeroCurve(**curve_info["svensson"])
print(f"fit error {curve_info['rmse_bp']:.1f} bp; 5y zero rate {curve.zero(5):.2%}, 10y {curve.zero(10):.2%}")
pd.DataFrame(shocks.bucket_table(), index=[b[0] for b in shocks.BUCKETS]).T

fit error 1.8 bp; 5y zero rate 0.10%, 10y 0.32%


,O/N,1M,3M,6M,9M,1Y,1.5Y,2Y,3Y,4Y,5Y,6Y,7Y,8Y,9Y,10Y,15Y,20Y,>20Y
parallel_up,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150
parallel_down,-150,-150,-150,-150,-150,-150,-150,-150,-150,-150,-150,-150,-150,-150,-150,-150,-150,-150,-150
steepener,-97,-96,-90,-81,-70,-61,-47,-31,-10,12,29,43,53,61,68,73,82,88,90
flattener,120,118,113,104,94,85,72,56,36,15,-2,-14,-25,-32,-39,-43,-52,-58,-60
short_up,150,148,144,137,128,121,110,97,80,63,49,38,30,23,18,14,7,2,0
short_down,-150,-148,-144,-137,-128,-121,-110,-97,-80,-63,-49,-38,-30,-23,-18,-14,-7,-2,0


## 4. Deposits: how much do client rates follow SARON?
Pass-through betas estimated on SNB client rates, June 2022 - December 2025.

In [5]:
assumptions = json.loads((ROOT / "results" / "assumptions.json").read_text())
pd.DataFrame(assumptions["pass_through"]).T[["beta", "r2", "months"]]

,beta,r2,months
sight_deposits,0.013081,0.721753,43
savings_deposits,0.346715,0.753166,43
mortgages_variable,0.10981,0.403059,43


## 5. ΔEVE and ΔNII of the model vs BCV

In [6]:
positions = build_positions(assumptions)
rows = {}
for name, shock in shocks.all_shocks().items():
    rows[name] = {"model ΔEVE": delta_eve(positions, curve, shock)[0],
                  "BCV ΔEVE": latest.loc[name, "delta_eve_chf_m"],
                  "model ΔNII": delta_nii(positions, shock)[0],
                  "BCV ΔNII": latest.loc[name, "delta_nii_chf_m"]}
comparison = pd.DataFrame(rows).T.round(0)
comparison

,model ΔEVE,BCV ΔEVE,model ΔNII,BCV ΔNII
parallel_up,-449.0,-467.0,96.0,78.0
parallel_down,454.0,488.0,-158.0,-3.0
steepener,-41.0,-138.0,-68.0,NaN
flattener,-49.0,40.0,47.0,NaN
short_up,-187.0,-120.0,71.0,NaN
short_down,191.0,125.0,-133.0,NaN


## 6. Outlier test: worst ΔEVE / Tier 1 against 15%

In [7]:
worst = comparison["model ΔEVE"].idxmin()
print(f"model:     {-comparison.loc[worst, 'model ΔEVE'] / tier1:.1%} ({worst})")
print(f"published: {-latest.delta_eve_chf_m.min() / tier1:.1%}")

model:     12.9% (parallel_up)
published: 13.4%


## 7. Sensitivity, prepayment and hedge
See `scripts/04_nmd_prepayment_hedge.py`. Key numbers:

In [8]:
sens = json.loads((ROOT / "results" / "sensitivity_results.json").read_text())
hedge = sens["hedge"]
print("ΔEVE parallel up if the deposit stable share is halved / x1.5:")
for k, v in sens["nmd_one_at_a_time_parallel_up"].items():
    print(f"  {k:40s} {v:7.0f}")
print(f"worst ratio without BCV's swaps: {hedge['worst_without_swaps'][1]:.1%}")
print(f"extra 5y payer swap to reach 10%: CHF {hedge['notional_chf_m']:,.0f} m at {hedge['swap_rate']:.2%}")

ΔEVE parallel up if the deposit stable share is halved / x1.5:
  sight_deposits stable share x0.5            -674
  sight_deposits stable share x1.5            -274
  savings_deposits stable share x0.5          -618
  savings_deposits stable share x1.5          -280
worst ratio without BCV's swaps: 29.9%
extra 5y payer swap to reach 10%: CHF 1,393 m at 0.10%
